# Phase 7: Reverse Distillation timing test

**CRISP-DM stage:** Modelling

Reverse Distillation is the only method here that **learns over many rounds**. One round through all good training photos is called an **epoch**. Before choosing how many epochs to train (a major decision), this notebook measures what one epoch costs.

**What it does, per product:** build the model as anomalib does, train for 3 epochs on the good training photos, and record each epoch's time, average loss and peak GPU memory. The first epoch includes set-up work, so epochs 2 and 3 give the fair time.

**What it does not do:** score any test photo. Half A and half B are not read. Only one batch of good validation photos is used, to check that heat maps come out with the right shape.

**Runs on:** Kaggle with the GPU and internet on. On the laptop it only checks the code, with an untrained teacher, tiny photos and the sample folder.

In [ ]:
!pip install anomalib==2.7.0

In [ ]:
import random
import time
from pathlib import Path

import numpy
import torch
import yaml
from anomalib.data import MVTecAD2
from anomalib.models import ReverseDistillation
from anomalib.models.image.reverse_distillation.loss import ReverseDistillationLoss
from anomalib.models.image.reverse_distillation.torch_model import ReverseDistillationModel
from torch.utils.data import DataLoader

USING_GPU = torch.cuda.is_available()
if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    PRETRAINED = True
    IMAGE_SIZE = 256
    TIMING_EPOCHS = 3
else:
    # The laptop only proves the code runs
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    PRETRAINED = False
    IMAGE_SIZE = 64
    TIMING_EPOCHS = 1

# Finds the C8 settings file: the private dataset on Kaggle, or the project folder on the laptop
settings_file = None
if USING_GPU:
    search_folder = Path("/kaggle/input")
else:
    search_folder = Path("configs")
for found in search_folder.rglob("C8_*.yaml"):
    settings_file = found
if settings_file is None:
    raise FileNotFoundError("Could not find the C8 settings file")
with open(settings_file, encoding="utf-8") as file:
    method = yaml.safe_load(file)["method"]
BATCH_SIZE = method["train_batch_size"]
SEED = 42
print("Settings:", settings_file.name, "| network:", method["backbone"], "| layers:", method["layers"],
      "| batch size:", BATCH_SIZE, "| learning rate:", method["learning_rate"], "| betas:", method["betas"])


def set_seed(seed):
    """Fix the random numbers used by Python, NumPy and PyTorch."""
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)


def peak_gb():
    """Peak GPU memory since the last reset, in GB. Zero on the laptop."""
    if USING_GPU:
        return round(torch.cuda.max_memory_allocated() / 1e9, 3)
    return 0.0


# anomalib's default image preparation for this method: resize to a square, then ImageNet normalization
recipe = ReverseDistillation.configure_pre_processor(image_size=(IMAGE_SIZE, IMAGE_SIZE)).transform
rows = []

for product_name in ["sheet_metal", "vial"]:
    product_folder = None
    for train_folder in INPUT_FOLDER.rglob("train"):
        if train_folder.parent.name == product_name:
            product_folder = train_folder.parent
    datamodule = MVTecAD2(root=product_folder.parent, category=product_name, num_workers=0)
    datamodule.setup()
    train_photos = len(datamodule.train_data)
    print("===", product_name, "| good training photos:", train_photos)

    set_seed(SEED)
    model = ReverseDistillationModel(backbone=method["backbone"], input_size=(IMAGE_SIZE, IMAGE_SIZE),
                                     layers=method["layers"], anomaly_map_mode=method["anomaly_map_mode"],
                                     pre_trained=PRETRAINED).to(DEVICE)
    # Only the student (bottleneck and decoder) learns. The teacher (encoder) stays frozen
    student_parameters = list(model.decoder.parameters()) + list(model.bottleneck.parameters())
    optimizer = torch.optim.Adam(student_parameters, lr=method["learning_rate"], betas=tuple(method["betas"]))
    loss_function = ReverseDistillationLoss()
    # Shuffled each epoch, as in normal training, with a fixed seed so it can be repeated
    shuffler = torch.Generator()
    shuffler.manual_seed(SEED)
    loader = DataLoader(datamodule.train_data, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=datamodule.train_data.collate_fn, generator=shuffler)

    for epoch in range(1, TIMING_EPOCHS + 1):
        if USING_GPU:
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        model.train()
        started = time.perf_counter()
        loss_total = 0.0
        batches = 0
        for batch in loader:
            images = recipe(batch.image).to(DEVICE)
            teacher_features, student_features = model(images)
            loss = loss_function(teacher_features, student_features)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_total = loss_total + loss.item()
            batches = batches + 1
        if USING_GPU:
            torch.cuda.synchronize()
        seconds = time.perf_counter() - started
        row = {"product": product_name, "epoch": epoch, "seconds": round(seconds, 2),
               "average_loss": round(loss_total / batches, 5), "peak_gpu_gb": peak_gb(),
               "train_photos": train_photos, "batch_size": BATCH_SIZE}
        rows.append(row)
        print(row)

    # Shape check on one batch of good validation photos: one heat map per photo, at the image size
    model.eval()
    validation_loader = DataLoader(datamodule.val_data, batch_size=BATCH_SIZE, collate_fn=datamodule.val_data.collate_fn)
    validation_batch = next(iter(validation_loader))
    with torch.no_grad():
        output = model(recipe(validation_batch.image).to(DEVICE))
    print("Heat map shape:", tuple(output.anomaly_map.shape), "| score shape:", tuple(output.pred_score.shape))
    expected = (len(validation_batch.image), 1, IMAGE_SIZE, IMAGE_SIZE)
    if tuple(output.anomaly_map.shape) != expected:
        raise ValueError("Unexpected heat map shape")
    del model, optimizer
    if USING_GPU:
        torch.cuda.empty_cache()

results_folder = Path("results")
results_folder.mkdir(exist_ok=True)
with open(results_folder / "reverse_distillation_timing.csv", "w") as file:
    file.write(",".join(rows[0].keys()) + "\n")
    for row in rows:
        values = []
        for key in row:
            values.append(str(row[key]))
        file.write(",".join(values) + "\n")
print("Finished")